In [1]:
import sys; sys.path.insert(0, '..')
from fantasy.yahoo_client import get_my_roster, get_current_matchup
from fantasy.nba_stats import get_week_games_detail, get_matchup_factor, batch_get_player_stats, get_player_game_log
from fantasy.analysis import project_player, consistency_score, usage_spike
from fantasy.llm import build_start_sit_prompt, ask_gemini
import pandas as pd


In [2]:
matchup = get_current_matchup()
week_start, week_end = matchup['week_start'], matchup['week_end']
my_roster = get_my_roster()

with_stats = batch_get_player_stats(my_roster)
players = []
for p in with_stats:
    detail = get_week_games_detail(p['team_abbr'], week_start, week_end)
    mf = get_matchup_factor(p['team_abbr'], week_start, week_end)
    players.append({
        **p,
        'games_remaining': detail['total'],
        'games_detail': detail,
        'matchup_factor': mf,
    })

print(f'Loaded {len(players)} players')


In [3]:
for p in players:
    if p['stats'] is not None:
        p['projected'] = project_player(
            p['stats'], p['games_detail'], p.get('status', ''),
            matchup_factor=p.get('matchup_factor', 1.0)
        )
        # Consistency: std dev of PTS over last 15 games
        logs = get_player_game_log(p['name'])
        consistency = consistency_score(logs)
        p['std_pts'] = consistency.get('PTS', 0.0)
        # Scale std dev to weekly total (per-game std × sqrt(games))
        import math
        g = p['games_remaining'] or 1
        p['std_pts'] = round(p['std_pts'] * math.sqrt(g), 1)
        p['usg_spike'] = usage_spike(p['stats'])
    else:
        p['projected'] = {col: 0.0 for col in ['PTS','REB','AST','STL','BLK','TOV','FGM','FGA','FG3M','FTM','FTA']}
        p['std_pts'] = 0.0
        p['usg_spike'] = 0.0

players.sort(key=lambda p: p['projected'].get('PTS', 0), reverse=True)


In [4]:
rows = []
for p in players:
    proj = p['projected']
    pts = proj.get('PTS', 0)
    std = p['std_pts']
    b2b = '⚠️ B2B' if p.get('games_detail', {}).get('b2b_count', 0) > 0 else ''
    usg = p.get('usg_spike', 0.0)
    mf = p.get('matchup_factor', 1.0)
    rows.append({
        'Player': p['name'],
        'Status': p['status'] or 'Active',
        'G': p['games_remaining'],
        'Proj PTS': round(pts, 1),
        'Floor': round(max(0, pts - std), 1),
        'Ceil': round(pts + std, 1),
        'Matchup': round(mf, 3),
        'USG%Δ': f'+{usg:.1f}' if usg > 0 else f'{usg:.1f}',
        'B2B': b2b,
    })
print(pd.DataFrame(rows).to_string(index=False))


In [6]:
import os
from dotenv import load_dotenv
load_dotenv('/mnt/c/Users/louis/Documents/dev/.env', override=True)

prompt = build_start_sit_prompt(players)
advice = ask_gemini(prompt)
print('\n=== TODAY\'S LINEUP CARD ===\n')
print(advice)
